# AI Twin — Voice Clone Smoke Test (EN + HI)
**What this does:** zero-shot clones your voice from your own samples and renders
the opening lines of the pilot script in **English and Hindi**, using
**Chatterbox Multilingual** (MIT license, by Resemble AI). No training step.

**Cost: $0.** Runs on Colab's free GPU.

**Before you run:**
1. Runtime → Change runtime type → **T4 GPU**
2. Have your cleaned WAVs ready to upload: `english_162s.wav`, `hindi_49s.wav`
   (and `english_72s.wav` if prepared)

**Note:** if the install cell asks you to restart the runtime (torch version
change), do Runtime → Restart session, then run all cells again — the install
will be instant the second time.

In [ ]:
# Cell 1 — GPU check
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise SystemExit("No GPU — set Runtime > Change runtime type > T4 GPU")

In [ ]:
# Cell 2 — Install Chatterbox (one-time, ~2-3 min)
!pip install -q chatterbox-tts

In [ ]:
# Cell 3 — Upload your voice samples
from google.colab import files
import os
os.makedirs("refs", exist_ok=True)
print("Upload english_162s.wav and hindi_49s.wav (from your prepared/voice/ folder)")
uploaded = files.upload()
for name in uploaded:
    os.rename(name, f"refs/{name}")
print("Uploaded:", os.listdir("refs"))

In [ ]:
# Cell 4 — Trim reference clips (~15s each; zero-shot works best on a short clean reference)
import torchaudio

def trim_ref(in_path, out_path, start_s=2.0, dur_s=15.0):
    wav, sr = torchaudio.load(in_path)
    a, b = int(start_s * sr), int((start_s + dur_s) * sr)
    torchaudio.save(out_path, wav[:, a:b], sr)
    print(f"{out_path}: {dur_s}s @ {sr}Hz")

trim_ref("refs/english_162s.wav", "refs/ref_en.wav")
trim_ref("refs/hindi_49s.wav",   "refs/ref_hi.wav", start_s=1.0)

In [ ]:
# Cell 5 — Load the multilingual model (V3 if available, else V2)
from chatterbox.mtl_tts import ChatterboxMultilingualTTS

try:
    model = ChatterboxMultilingualTTS.from_pretrained(device="cuda", t3_model="v3")
    print("Loaded Chatterbox Multilingual V3")
except TypeError:
    model = ChatterboxMultilingualTTS.from_pretrained(device="cuda")
    print("Loaded Chatterbox Multilingual (V2 — older package version)")

In [ ]:
# Cell 6 — ENGLISH test: pilot script opening, cloned from your English sample
import torchaudio as ta

en_lines = (
    "There's a river running through my whole story. The Yamuna. "
    "On one bank, Ewing Christian College, where I studied. "
    "On the other, land that grew into the place that became SHUATS. Where I did my PhD."
)
# Defaults: exaggeration=0.5, cfg_weight=0.5. Lower cfg (~0.3) = slower, more deliberate pace.
wav = model.generate(en_lines, language_id="en",
                     audio_prompt_path="refs/ref_en.wav",
                     exaggeration=0.5, cfg_weight=0.4)
ta.save("twin_en_test.wav", wav, model.sr)

from IPython.display import Audio, display
print("ENGLISH:")
display(Audio("twin_en_test.wav"))

In [ ]:
# Cell 7 — HINDI test: same beats, Devanagari, cloned from your HINDI sample
# (Reference language should match the text language. If you hear an English
#  accent bleeding in, set cfg_weight=0 — official mitigation.)
hi_lines = (
    "मेरी पूरी कहानी में एक नदी बहती है। यमुना। "
    "एक किनारे पर इविंग क्रिश्चियन कॉलेज, जहाँ मैंने पढ़ाई की। "
    "दूसरे किनारे पर वह ज़मीन, जो आगे चलकर शुआट्स बनी। जहाँ मैंने अपनी पीएचडी की।"
)
wav_hi = model.generate(hi_lines, language_id="hi",
                        audio_prompt_path="refs/ref_hi.wav",
                        exaggeration=0.5, cfg_weight=0.4)
ta.save("twin_hi_test.wav", wav_hi, model.sr)
print("HINDI:")
display(Audio("twin_hi_test.wav"))

In [ ]:
# Cell 8 — Download both results
files.download("twin_en_test.wav")
files.download("twin_hi_test.wav")
print("Done. Judge with your own ears — then we decide: good enough, tune settings, or try the dedicated Hindi finetune (ResembleAI/Chatterbox-Multilingual-hi).")

## What to listen for
- **Is it you?** Timbre, pace, the way you land sentence endings.
- **Hindi test:** natural Hindi rhythm vs. "English speaker reading Hindi".
- **Artifacts:** metallic edges, swallowed syllables, weird pauses.

## Knobs if something is off
| Problem | Fix |
|---|---|
| Too fast / rushed | `cfg_weight=0.3` |
| Flat, not expressive | `exaggeration=0.7`, `cfg_weight=0.3` |
| English accent in Hindi | `cfg_weight=0` on the Hindi cell |
| Doesn't sound like you | try a different 15s slice in Cell 4 (clearest, most animated stretch) |

**FYI:** every Chatterbox output carries Resemble's imperceptible Perth watermark —
good for responsible-AI provenance, worth mentioning in the tutorial.